# 13. The requirements scoreboard: MAUT over the hierarchy

Requirements do more than pass or fail. Stakeholders care about *how
well* a design satisfies them -- and not every requirement matters
equally. This tutorial layers **multi-attribute utility theory (MAUT)**
over the requirements hierarchy with
`longeron.analysis.scoreboard`:

- every leaf requirement maps a raw measured value (with its declared
  **unit**) onto a **utility** in [0, 1] through a declared *utility
  shape*;
- parents aggregate their children's utilities by **importance
  weight** (simple additive weighting by default; weakest-link and
  geometric strategies are built in, and any callable plugs in);
- the root aggregate is the design's overall **score**;
- one interactive widget renders it all as a **treemap or Voronoi
  tessellation** where *area is importance* and *color is utility*.

The declarations live **in the model** -- plain attribute usages the
grammar already parses on requirements -- so the scoreboard needs no
Python-side configuration to score a design (Python overrides exist for
exploration).

Prerequisites: tutorial 3 (requirements and constraints); tutorial 7
helps for the trade-study bridge at the end. Scoring runs on the
interpreter alone; only the widget needs the `viz` extra (anywidget).

In [ ]:
import longeron
from longeron.analysis.scoreboard import architecture_values, scoreboard

## Weights, utility shapes, measures, and units -- in the model

Four reserved attribute names on a requirement usage (or on its typing
definition: typed usages inherit the declarations, and their own
declarations override inherited ones):

| attribute | meaning | default |
|---|---|---|
| `weight : Real` | importance among siblings (the widget's area share) | `1.0` |
| `utility : String` | utility shape name | `"step"` (pass/fail) |
| `measure : Real` | expression producing the raw measured value | the requirement's own `require constraint` check |
| `unit : String` | measurement unit of the raw value -- display-only, shown after `raw` in tooltips and tables; the ramp/target anchors are read in the same unit | `""` (unitless) |

The shape vocabulary, with its anchor attributes:

- `"larger-is-better"` / `"smaller-is-better"` -- linear between
  `ramp0` (utility 0) and `ramp1` (utility 1), clamped outside;
  orientation is validated;
- `"ramp"` -- the same ramp, either orientation allowed;
- `"target-is-best"` -- 1 at `target`, falling to 0 at `limit` away;
- `"step"` -- pass/fail (exactly 1 or 0).

A requirement with no measure and no constraints -- or whose `assume`
does not hold -- is **unmeasured**: NaN utility, excluded from its
parent's aggregation, hatched grey in the widget.

**Units.** The design point below carries *real* SysML quantity values:
`12.0 [SI::min]` parses today (the bracketed measurement reference
annotates the literal) and evaluates to its magnitude, and the vendored
standard library ships the `Quantities`/`ISQ`/`SI` packages, so
`longeron.add_standard_library(model)` resolves `SI::min` to the actual
`DurationUnit`. The scoreboard itself never converts anything: the
`unit` attribute is the pragmatic display convention (a pint-backed
units integration is designed separately), and the ramp anchors are
simply read in the same unit as the measure.

The model below is a small reconnaissance quadcopter: a weighted
mission requirement hierarchy scoring the *current design point* (the
package-level attributes), plus a component catalog we will trade over
at the end. The endurance branch runs **three levels deep** below the
root (`mission > performance > endurance > hover/cruise`) -- exactly
what the widget's zoom navigation is made for.

In [ ]:
SRC = """
package ScoutUAV {
    // the current design point: what the requirements score by default
    // (real SysML quantity values: the bracketed measurement reference
    // parses and annotates; evaluation yields the magnitude)
    attribute hoverTime : Real = 12.0 [SI::min];   // full-hover endurance
    attribute cruiseTime : Real = 20.0 [SI::min];  // full-cruise endurance
    attribute radius_km : Real = 8.5 [SI::km];
    attribute totalMass : Real = 1.62 [SI::kg];
    attribute unitCost : Real = 950.0;             // USD -- no SI unit
    attribute noise_dB : Real = 68.0 [SI::dB];     // at 50 m

    requirement mission {
        requirement performance {
            attribute weight : Real = 3.0;
            requirement endurance {              // three levels below the
                attribute weight : Real = 3.0;   // root: mission > performance
                requirement hoverEndurance {     //   > endurance > *
                    attribute utility : String = "larger-is-better";
                    attribute ramp0 : Real = 5.0;
                    attribute ramp1 : Real = 20.0;
                    attribute measure : Real = hoverTime;
                    attribute unit : String = "min";
                }
                requirement cruiseEndurance {
                    attribute weight : Real = 2.0;
                    attribute utility : String = "larger-is-better";
                    attribute ramp0 : Real = 10.0;
                    attribute ramp1 : Real = 30.0;
                    attribute measure : Real = cruiseTime;
                    attribute unit : String = "min";
                }
            }
            requirement radius {
                attribute weight : Real = 2.0;
                attribute utility : String = "larger-is-better";
                attribute ramp0 : Real = 3.0;
                attribute ramp1 : Real = 12.0;
                attribute measure : Real = radius_km;
                attribute unit : String = "km";
            }
        }
        requirement affordability {
            attribute weight : Real = 2.0;
            requirement cost {
                attribute utility : String = "smaller-is-better";
                attribute ramp0 : Real = 1500.0;
                attribute ramp1 : Real = 500.0;
                attribute measure : Real = unitCost;
                attribute unit : String = "USD";
            }
        }
        requirement operability {
            attribute weight : Real = 2.0;
            requirement mass {
                attribute weight : Real = 2.0;
                attribute utility : String = "smaller-is-better";
                attribute ramp0 : Real = 2.5;
                attribute ramp1 : Real = 1.0;
                attribute measure : Real = totalMass;
                attribute unit : String = "kg";
            }
            requirement regulatory {
                require constraint { totalMass <= 25.0 }
            }
            requirement quiet {
                attribute utility : String = "target-is-best";
                attribute target : Real = 60.0;
                attribute limit : Real = 15.0;
                attribute measure : Real = noise_dB;
                attribute unit : String = "dB";
            }
            requirement futureProofing;  // deliberately unmeasured
        }
    }
}

package ScoutCatalog {
    variation part def BatteryChoice {
        variant part packLight {
            attribute mass : Real = 0.45; attribute wh : Real = 55.0;
            attribute cost : Real = 180.0;
        }
        variant part packMax {
            attribute mass : Real = 0.75; attribute wh : Real = 99.0;
            attribute cost : Real = 260.0;
        }
    }
    variation part def MotorChoice {
        variant part stdMotor {
            attribute mass : Real = 0.062; attribute cost : Real = 38.0;
            attribute eff : Real = 0.85;
        }
        variant part proMotor {
            attribute mass : Real = 0.048; attribute cost : Real = 72.0;
            attribute eff : Real = 0.93;
        }
    }
    part def Scout {
        attribute baseMass : Real = 0.9;      // airframe + avionics + payload
        attribute baseCost : Real = 420.0;
        part battery : BatteryChoice;
        part motors : MotorChoice[4];
        attribute totalMass : Real = baseMass + battery.mass + 4.0 * motors.mass;
        attribute unitCost : Real = baseCost + battery.cost + 4.0 * motors.cost;
        // two endurance profiles: hover draws ~165 W/kg, cruise ~105 W/kg
        attribute hoverTime : Real =
            battery.wh * motors.eff / (totalMass * 165.0) * 60.0;
        attribute cruiseTime : Real =
            battery.wh * motors.eff / (totalMass * 105.0) * 60.0;
        attribute radius_km : Real = cruiseTime * 0.6 / 2.0;
        attribute noise_dB : Real = 50.0 + 9.0 * totalMass;
    }
}
"""
model = longeron.loads(SRC)

## Score the current design

`scoreboard(model)` finds the root requirement usages, evaluates every
`measure` through the interpreter, maps raws onto utilities, and
aggregates up the hierarchy. `str()` renders the whole decomposition;
`.score` is the root aggregate and `.table()` the same rows as data.

In [ ]:
sb = scoreboard(model)
print(sb)

Reading the table: `hoverEndurance` measures 12 min on a 5 -> 20 min
ramp (utility 0.47), `cruiseEndurance` 20 min on a 10 -> 30 min ramp
(utility 0.50), and their weighted mean is the `endurance` aggregate;
`regulatory` is a pass/fail check that passes; `futureProofing` is
unmeasured, so it is excluded from `operability`'s aggregate (its
weight simply does not count). Declared units print right after the raw
values. Every group's `aggregate` is the weight-normalized sum of its
measured children, and the root's aggregate is the score.

## The scoreboard widget: area = importance, color = utility

Each cell is a leaf requirement; its **area** is its weight share
(recursively, so sibling areas subdivide their parent's area) and its
**color** is its utility on a perceptual red -> yellow -> green ramp.
Unmeasured cells are hatched grey.

- **hover** a cell for the qualified name, weight and share, raw value
  with its declared unit, and utility (`raw 12 min · larger-is-better`);
- **click** selects (the `selected` trait -- more below);
- **double-click** a group cell to *zoom into it*: the subtree
  re-tessellates to fill the whole canvas, and a breadcrumb bar appears
  above the canvas tracking the zoom path (each crumb zooms back out;
  Esc steps out one level). Double-clicking a leaf zooms to its parent
  group. Zooming is pure navigation -- it never changes what is
  collapsed;
- the **▸/▾ twist** at a group's top-left corner *collapses or expands
  the group in place*: a collapsed group becomes one cell whose area is
  the subtree's total share and whose color is the subtree
  **aggregate**.

Try the deep branch on the widget below: **double-click** `endurance`
to zoom into it, watch the breadcrumb bar appear, hover
`hoverEndurance` for its unit-bearing tooltip, then press **Esc** to
step back out.

In [ ]:
board = sb.widget()
board

Collapse is also scriptable -- the `collapsed` trait lists collapsed
subtree qnames (the widget below starts with `performance` collapsed
into a single aggregated cell):

In [ ]:
folded = sb.widget(collapsed=["ScoutUAV::mission::performance"])
folded

## The Voronoi tessellation

The same scoreboard, tessellated as a Voronoi treemap (organic cells,
same area/color semantics; computed by the vendored
[d3-voronoi-treemap](https://github.com/Kcnarf/d3-voronoi-treemap),
BSD-3-Clause). The iteration is seeded, so the layout is deterministic
for a given `seed`. All the interactions carry over.

In [ ]:
voronoi = sb.widget("voronoi")
voronoi

## Aggregation strategies

Simple additive weighting (`"saw"`) is the MAUT default, but it happily
trades a terrible requirement against a great one. `"min"` (weakest
link) and `"geometric"` (weighted geometric mean) punish imbalance
progressively harder -- and any callable over `(weight, utility)` pairs
plugs in as a custom strategy.

In [ ]:
for strategy in ("saw", "min", "geometric"):
    print(f"{strategy:>12}: {scoreboard(model, aggregation=strategy).score:.3f}")


def weakest_two(children):
    """A custom Aggregator: the mean of the two weakest utilities."""
    worst = sorted(utility for _, utility in children)[:2]
    return sum(worst) / len(worst)


print(f"{'weakest_two':>12}: {scoreboard(model, aggregation=weakest_two).score:.3f}")

## What-if: injecting measured values

`values=` overrides raw measurements without touching the model: keys
match requirement qualified names, requirement names, or -- for plain
identifiers -- the free names inside `measure` expressions and
constraint bodies. A hypothetical long-range battery:

In [ ]:
upgraded = scoreboard(model, values={"hoverTime": 16.0, "cruiseTime": 27.0, "unitCost": 1150.0})
print(f"current design: {sb.score:.3f}")
print(f"long-range battery: {upgraded.score:.3f}")
assert upgraded.score > sb.score  # endurance gain outweighs the cost hit

## The trade-study bridge

That last form is exactly what `architecture_values` produces from a
trade-study architecture: `trades.Architecture.metrics` maps derived
attribute names to interpreter-exact values, and the `Scout` catalog's
derived attributes deliberately share their names with the requirement
measures. Ranking every catalog mix by its scoreboard score turns the
trade study's *feasible* answer into a *preferred* one:

In [ ]:
from longeron.analysis.trades import TradeStudy

study = TradeStudy(model, "ScoutCatalog::Scout")
ranked = sorted(
    study.all_architectures(),
    key=lambda arch: scoreboard(model, values=architecture_values(arch)).score,
    reverse=True,
)
for arch in ranked:
    score = scoreboard(model, values=architecture_values(arch)).score
    print(f"{score:.3f}  {arch}")
best = ranked[0]

In [ ]:
print(scoreboard(model, values=architecture_values(best)))

The big pack with premium motors wins on endurance despite the cost
hit -- and the table shows exactly where the remaining utility now
leaks: with the endurance branch nearly saturated, `mass` (the big
pack drags the mix to 1.84 kg) and `cost` are the weakest measured
leaves.

## Linked selection

Clicking a cell writes the requirement's qualified name to the
`selected` trait -- the same observer idiom as the diagram and 3D
widgets, so a scoreboard can join their linked-selection loops.
Selection also works kernel-side:

In [ ]:
hits = []
board.on_select(hits.append)
board.selected = ["ScoutUAV::mission::performance::endurance::cruiseEndurance"]
print(hits)

## Deep hierarchies: `max_depth` and scripted navigation

Zoom, breadcrumbs, and twists come into their own when trees grow past
what one canvas can label. The second view-state knob, **`max_depth`**,
windows the render depth below the *current zoom root*: deeper levels
draw as aggregate cells -- the same look as collapsed (aggregate color,
subtree area, ▸) *without* entering the `collapsed` trait -- and
zooming in reveals the next `max_depth` levels. Like zoom, it is pure
navigation and never changes the score.

With `max_depth=2` the endurance branch opens as one aggregate cell:
`hoverEndurance` and `cruiseEndurance` sit three levels down, past the
window. Double-click `endurance` to reveal them -- or script it, since
every navigation knob is a two-way trait:

In [ ]:
zoomable = sb.widget(max_depth=2)  # levels 3+ draw as aggregate cells
zoomable

In [ ]:
# drive the view from the kernel: zoom into the three-deep branch
zoomable.zoom_root = "ScoutUAV::mission::performance::endurance"
zoomable.max_depth = None  # and widen the window (None = unlimited)
print("zoomed into:", zoomable.zoom_root)
print(f"score (zoom is view state, never scoring state): {sb.score:.3f}")

## Where to go next

- Tutorial 7 covers the trade-study, MDAO, and SMT bridges the
  scoreboard composes with; `architecture_values` accepts any object
  with a `.metrics` dict.
- `weights=` and `utilities=` keyword overrides support quick
  sensitivity checks ("what if affordability mattered twice as
  much?") without editing the model -- but the model stays the source
  of truth.
- The full utility-shape and aggregation vocabulary is in the
  `longeron.analysis.scoreboard` reference page.